<a href="https://colab.research.google.com/github/sathundorn/Super-AI-Engineer-Season-6/blob/Hackaton4_601402/601402_%E0%B8%AA%E0%B8%98%E0%B8%A3%E0%B8%A3%E0%B8%94%E0%B8%A3_HouseRecognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import copy

In [ ]:
import seaborn as sns

In [ ]:
from google.colab import drive
import os

# 1. เชื่อมต่อ Google Drive
drive.mount('/content/drive')

# 2. กำหนด Path ของไฟล์ zip (แก้ path ให้ตรงกับโฟลเดอร์ที่คุณเก็บไฟล์ imagehack4.zip ไว้)
# สมมติว่าไฟล์อยู่ในหน้าแรกของ MyDrive
zip_path = "/content/drive/MyDrive/imagehack4/super-ai-engineer-season-6-individual-hackathon-house-recognition"

# โฟลเดอร์ที่เราจะแตกไฟล์ไปไว้ (ให้อยู่ในพื้นที่ของ Colab จะได้อ่านไฟล์เร็วขึ้น)
extract_path = "/content/drive/MyDrive/dataset4/"

# 3. แตกไฟล์ zip (-q คือเงียบๆ ไม่ต้องโชว์ชื่อไฟล์ทั้งหมด)
!unzip -q "{zip_path}" -d "{extract_path}"

print("แตกไฟล์เสร็จสิ้น!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/drive/MyDrive/dataset4/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
แตกไฟล์เสร็จสิ้น!


In [ ]:
# 4. กำหนด Path ของไฟล์ต่างๆ ที่เพิ่งแตกออกมา
train_dir = os.path.join(extract_path, "train")
test_dir = os.path.join(extract_path, "test")
train_csv_path = os.path.join(extract_path, "train.csv")
submission_csv_path = os.path.join(extract_path, "sample_submission.csv")

# 5. โหลดไฟล์ Annotations
df_train = pd.read_csv(train_csv_path)
df_sub = pd.read_csv(submission_csv_path)

print("ข้อมูล Train CSV:")
display(df_train.head())

ข้อมูล Train CSV:


,image_name,class
0,ChokChai4_img_13-7956791_100-6031267_a187-2159...,0
1,ChokChai4_img_13-7961753_100-6031881_a185-9785...,0
2,ChokChai4_img_13-7969811_100-5906061_a180-5812...,0
3,ChokChai4_img_13-7970811_100-5906071_a180-5812...,0
4,ChokChai4_img_13-7971811_100-5906081_a180-5812...,0


In [ ]:
class HouseRecognitionDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        # ดึงชื่อไฟล์จากคอลัมน์แรก
        img_id = str(self.annotations.iloc[index, 0])

        # เช็คว่าถ้ายังไม่มี .jpg ให้เติมเข้าไป แต่ถ้ามีแล้วให้ปล่อยผ่าน
        if not img_id.lower().endswith('.jpg'):
            img_id = f"{img_id}.jpg"

        # ประกอบ Path เต็ม
        img_path = os.path.join(self.root_dir, img_id)

        # เปิดรูปภาพและแปลงเป็น RGB
        image = Image.open(img_path).convert("RGB")

        # ดึงคำตอบ (Label) จากคอลัมน์ที่ 2
        y_label = int(self.annotations.iloc[index, 1])

        if self.transform:
            image = self.transform(image)

        return image, y_label

# 1. Transforms สำหรับ Training (จัดเต็ม Data Augmentation)
train_transforms = transforms.Compose([
    # --- เริ่มต้น Data Augmentation ---
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),   # สุ่มซูมภาพนิดหน่อยแล้วครอปให้ได้ขนาด 224x224
    transforms.RandomHorizontalFlip(p=0.5),                # สุ่มพลิกรูปภาพซ้าย-ขวา โอกาส 50%
    transforms.RandomRotation(degrees=15),                 # สุ่มหมุนภาพซ้ายขวาไม่เกิน 15 องศา
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # สุ่มปรับความสว่างและคอนทราสต์ให้ภาพ
    # ---------------------------------

    # แปลงภาพเพื่อเข้าโมเดล (ส่วนบังคับ)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 2. Transforms สำหรับ Testing / ทำนายผล (ห้ามใส่ Augmentation เด็ดขาด)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. กำหนด Path ตามข้อมูลในไฟล์ของคุณ
TRAIN_CSV = '/content/drive/MyDrive/dataset4/train.csv'
TRAIN_DIR = '/content/drive/MyDrive/dataset4/train/train'

# สร้าง Dataset สำหรับเทรน โดยเรียกใช้ train_transforms
train_dataset = HouseRecognitionDataset(
    csv_file=TRAIN_CSV,
    root_dir=TRAIN_DIR,
    transform=train_transforms  # <--- เปลี่ยนมาใช้ตัวที่มี Augmentation
)

# โหลดเข้า DataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

print(f"✅ โหลดข้อมูลสำเร็จ! จำนวนภาพใน Training Set: {len(train_dataset)} รูป พร้อมเปิดใช้งาน Data Augmentation แล้ว!")

✅ โหลดข้อมูลสำเร็จ! จำนวนภาพใน Training Set: 2953 รูป พร้อมเปิดใช้งาน Data Augmentation แล้ว!


In [ ]:
ls /content/drive/MyDrive/dataset4/train/train

In [ ]:
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

# เช็คว่ามี GPU ให้ใช้ไหม (ใน Colab อย่าลืมเปลี่ยน Runtime เป็น T4 GPU นะครับ)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"กำลังใช้งาน: {device}")


# อัปเกรดเป็น ResNet-50
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# 2. (ตัวเลือก) แช่แข็ง (Freeze) น้ำหนักของ Layer แรกๆ ไว้ เพื่อให้เทรนเร็วขึ้นในตอนต้น
for param in model.parameters():
    param.requires_grad = False

# 3. เปลี่ยน Layer สุดท้าย (Fully Connected) ให้เป็น 2 คลาส (0 และ 1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) # ตอนนี้จะเทรนแค่น้ำหนักส่วนนี้

# ย้ายโมเดลไปที่ GPU
model = model.to(device)

# 4. กำหนด Loss Function และ Optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # สังเกตว่าเราอัปเดตแค่ model.fc

กำลังใช้งาน: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s]


In [ ]:
# 1. Unfreeze ทั้งเครือข่าย (เปิดให้เทรนได้ทุก Layer)
for param in model.parameters():
    param.requires_grad = True

# 2. ปรับ Optimizer เป็น AdamW และลด LR ลง (1e-4 หรือ 1e-5)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# 3. ตั้งค่าจำนวน Epoch และ LR Scheduler
num_epochs = 15 # เทรนให้นานขึ้นได้เลยเพราะเรามี Scheduler กับ Checkpoint
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# 4. เรียกใช้ GradScaler สำหรับ Mixed Precision (AMP)
scaler = GradScaler()

# ตัวแปรสำหรับเก็บโมเดลที่ดีที่สุด
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

print("🔥 เริ่มต้นกระบวนการ True Fine-Tuning แบบจัดเต็ม...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # ล้างค่า Gradient
        optimizer.zero_grad()

        # 5. Forward pass แบบ AMP (Mixed Precision)
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # 6. Backward pass & Optimize แบบ AMP
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # เก็บค่าสถิติ
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # อัปเดต Learning Rate 1 ครั้งต่อ 1 Epoch
    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    current_lr = scheduler.get_last_lr()[0]

    print(f'Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}% | LR: {current_lr:.6f}')

    # 7. Model Checkpointing: เก็บโมเดลที่ดีที่สุดไว้
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_house_model.pth') # เซฟไฟล์ลงเครื่องเผื่อไว้
        print(f"   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น {best_acc:.2f}%")

print("="*50)
print(f'🏆 เทรนเสร็จสิ้น! ความแม่นยำสูงสุด (Train Acc) ที่ทำได้: {best_acc:.2f}%')

# 8. โหลด Weights ของ Epoch ที่ดีที่สุดกลับเข้าโมเดล เตรียมนำไป Test
model.load_state_dict(best_model_wts)
print("-> ✅ โหลด Best Weights กลับเข้าโมเดลเรียบร้อย พร้อมเอาไปทำนาย Submission แล้วครับ!")

/tmp/ipykernel_1128/1009466381.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


🔥 เริ่มต้นกระบวนการ True Fine-Tuning แบบจัดเต็ม...


/tmp/ipykernel_1128/1009466381.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/15] | Loss: 0.2263 | Acc: 98.88% | LR: 0.000099
   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น 98.88%
Epoch [2/15] | Loss: 0.2245 | Acc: 98.85% | LR: 0.000096
Epoch [3/15] | Loss: 0.2121 | Acc: 99.56% | LR: 0.000090
   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น 99.56%
Epoch [4/15] | Loss: 0.2188 | Acc: 99.22% | LR: 0.000083
Epoch [5/15] | Loss: 0.2128 | Acc: 99.49% | LR: 0.000075
Epoch [6/15] | Loss: 0.2107 | Acc: 99.46% | LR: 0.000065
Epoch [7/15] | Loss: 0.2075 | Acc: 99.70% | LR: 0.000055
   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น 99.70%
Epoch [8/15] | Loss: 0.2069 | Acc: 99.70% | LR: 0.000045
Epoch [9/15] | Loss: 0.2045 | Acc: 99.86% | LR: 0.000035
   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น 99.86%
Epoch [10/15] | Loss: 0.2033 | Acc: 99.90% | LR: 0.000025
   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น 99.90%
Epoch [11/15] | Loss: 0.2013 | Acc: 99.97% | LR: 0.000017
   🌟 เจอโมเดลที่ดีที่สุดใหม่! อัปเดต Best Acc เป็น 99.97%
Epoch [12/15] | Loss: 0

In [ ]:
import torch.nn.functional as F

model.eval()
predictions = []

print(f"กำลังทำนายผลพร้อมใช้ TTA จากรูปภาพทั้งหมด {len(df_sub)} รูป...")

for index, row in df_sub.iterrows():
    img_id = str(row[id_col])
    if not img_id.lower().endswith('.jpg'):
        img_filename = f"{img_id}.jpg"
    else:
        img_filename = img_id

    img_path = os.path.join(test_dir, img_filename)

    if not os.path.exists(img_path):
        predictions.append({id_col: row[id_col], answer_col: 0})
        continue

    # เปิดและแปลงรูปภาพ
    image = Image.open(img_path).convert("RGB")

    # 1. ภาพต้นฉบับ
    image_tensor_1 = test_transforms(image).unsqueeze(0).to(device)

    # 2. ภาพกลับซ้ายขวา (Horizontal Flip) สำหรับ TTA
    image_tensor_2 = torch.flip(image_tensor_1, dims=[3])

    # ทายผลด้วยโมเดลทั้ง 2 รูปแบบ
    with torch.no_grad():
        output_1 = F.softmax(model(image_tensor_1), dim=1) # ได้ค่าความน่าจะเป็น
        output_2 = F.softmax(model(image_tensor_2), dim=1)

        # นำความน่าจะเป็นมาเฉลี่ยกัน (Ensemble by Averaging)
        avg_output = (output_1 + output_2) / 2.0

        # เลือกคลาสที่คะแนนโหวตเฉลี่ยสูงที่สุด
        _, predicted_class = torch.max(avg_output, 1)

    predictions.append({id_col: row[id_col], answer_col: predicted_class.item()})

final_submission = pd.DataFrame(predictions)
final_submission.to_csv('submission_tta_resnet50.csv', index=False)
print("✅ บันทึกไฟล์ TTA เรียบร้อย! นำไปส่งได้เลยครับ")

กำลังทำนายผลพร้อมใช้ TTA จากรูปภาพทั้งหมด 1550 รูป...
✅ บันทึกไฟล์ TTA เรียบร้อย! นำไปส่งได้เลยครับ
